# HYPER-3, 6 — Running it: the interim that stops an arm, and the readout

The design is fixed. This notebook runs it: eight safety reviews on the trial that
actually happened, twelve contrasts each time, `monitor` against the boundaries from
notebook 5, and then whatever readout survives.

What happens is the thing the whole case study was built around. The 40 mg arm looks
unremarkable at every arm-level review and is stopped in the oldest age band at the
*first* one, on three eighths of the information — with a third of the units planned for
that arm never randomized to it and the rest taken off it months early.

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import hyper3 as h
from axiom.core import BASES, D, Interval, LedgerLine, eti, interval, wald
from axiom.design import (
    Boundary, LookSchedule, MonitoringPath, StoppingRule, alpha_spending, crossing_probabilities,
    difference_se, harm_boundary, information_fractions, monitor, operating_characteristics,
)
from axiom.identify import CausalGraph, identify, ols
from scipy import stats as sps
from axiom.meta import ForestData, Heterogeneity, forest_data, heterogeneity, random_effects
from axiom.viz import forest, funnel
from axiom.meta import funnel_data

BASES.declare("mass", symbol="M")
SD_WINDOW, HARM_MARGIN, HARM_PROBABILITY = 6.0, 2.0, 0.95
MASS = 0.95
trial = h.trial(seed=20260821)

LOOK_WEEKS = list(range(12, 24, 2))
INFORMATION = information_fractions([h.N_UNITS * (w - 6) / 16 for w in LOOK_WEEKS], h.N_UNITS)
schedule = LookSchedule(labels=tuple(f"week_{w}" for w in LOOK_WEEKS), information=INFORMATION)

N_ARM = {arm: int(h.N_UNITS * k / h.BLOCK) for arm, k in h.ALLOCATION.items()}
N_CELL = {(s, a): int(h.N_UNITS * h.STRATUM_SHARE[s] * k / h.BLOCK)
          for s in h.STRATA for a, k in h.ALLOCATION.items()}
CONTRASTS = [("all", arm) for arm in h.ARMS[1:]] + \
            [(s, arm) for s in h.STRATA for arm in h.ARMS[1:]]

rules: dict[tuple[str, str], StoppingRule] = {}
for stratum, arm in CONTRASTS:
    if stratum == "all":
        n_dose, n_control = N_ARM[arm], N_ARM["standard_of_care"]
    else:
        n_dose, n_control = N_CELL[(stratum, arm)], N_CELL[(stratum, "standard_of_care")]
    se = difference_se(n_dose + n_control, sd=SD_WINDOW, allocation=n_dose / (n_dose + n_control))
    rules[(stratum, arm)] = StoppingRule(
        name=f"harm|{stratum}|{arm}", looks=schedule,
        boundaries=(harm_boundary(HARM_PROBABILITY, INFORMATION, margin=HARM_MARGIN,
                                  se_at_full_information=se),))
print(f"{len(rules)} monitored contrasts, {schedule.n_looks} scheduled safety reviews at "
      f"calendar weeks {LOOK_WEEKS}")
print("information fractions:", tuple(round(t, 3) for t in INFORMATION))

## 1. Six reviews, twelve contrasts

At each review the analysis is the one the protocol names: an ANCOVA of the safety-window
change on the treatment indicator, adjusting for baseline pressure (and for stratum when
the contrast is pooled). `Z = −effect / se`, signed so that positive is better, is what
`monitor` reads.

In [ ]:
def look_statistics(calendar_week: int) -> dict[tuple[str, str], tuple[float, float, int]]:
    """The (effect, se, n) each contrast would report at this review."""
    frame = h.look_frame(trial, calendar_week, endpoint="safety")
    out: dict[tuple[str, str], tuple[float, float, int]] = {}
    for stratum, arm in CONTRASTS:
        where = None if stratum == "all" else stratum
        rows = h.contrast_frame(frame, arm, stratum=where)
        estimate = ols(rows, "change", "treated", h.ancova_covariates(where))
        out[(stratum, arm)] = (estimate.estimate, estimate.se, estimate.n)
    return out


series = {week: look_statistics(week) for week in LOOK_WEEKS}
z_paths: dict[tuple[str, str], list[float]] = {key: [] for key in rules}
effects: dict[tuple[str, str], list[float]] = {key: [] for key in rules}
ses: dict[tuple[str, str], list[float]] = {key: [] for key in rules}
for week in LOOK_WEEKS:
    for key in rules:
        effect, se, _ = series[week][key]
        z_paths[key].append(-effect / se)
        effects[key].append(effect)
        ses[key].append(se)
first_look = h.look_frame(trial, LOOK_WEEKS[0])
smallest = min(int(h.contrast_frame(first_look, arm, stratum=None if s == "all" else s)
                   ["treated"].sum()) for s, arm in CONTRASTS)
print(f"at the first review every contrast is evaluable: the smallest dose arm among the "
      f"twelve has {smallest} units")

paths: dict[tuple[str, str], MonitoringPath] = {
    key: monitor(rules[key], z_paths[key], effects=effects[key], ses=ses[key]) for key in rules
}
rows = []
for (stratum, arm), path in paths.items():
    stop = path.stop
    rows.append({
        "stratum": "all units" if stratum == "all" else h.STRATUM_LABEL[stratum],
        "arm": h.ARM_LABEL[arm], "looks_taken": len(path.looks), "decision": path.decision,
        "stopped_week": "" if stop is None else LOOK_WEEKS[stop.look],
        "z_at_stop": np.nan if stop is None else stop.z,
        "lowest_z_seen": min(z_paths[(stratum, arm)][:len(path.looks)]),
        "z_if_it_had_continued": np.nan if stop is None else min(z_paths[(stratum, arm)]),
    })
monitoring = pd.DataFrame(rows)
print(monitoring.to_string(index=False))

One contrast crossed. Every other one ran the full six reviews, including the three
arm-level monitors — which is the whole point: 40 mg pooled over the trial population
never looked like anything worth stopping.

## 2. The plot the committee sees

In [ ]:
fig = make_subplots(rows=2, cols=2, shared_xaxes=True, subplot_titles=(
    "40 mg, all units", "40 mg, 25–35", "40 mg, 36–50", "40 mg, 51+"))
panels = [("all", 1, 1), ("age_25_35", 1, 2), ("age_36_50", 2, 1), ("age_51_plus", 2, 2)]
for stratum, row, col in panels:
    key = (stratum, "dose_40")
    boundary = rules[key].boundaries[0]
    taken = len(paths[key].looks)
    colour = "#5b6472" if stratum == "all" else h.STRATUM_COLOR[stratum]
    fig.add_trace(go.Scatter(x=list(INFORMATION) + list(INFORMATION)[::-1],
                             y=list(boundary.z) + [-9.0] * len(INFORMATION),
                             fill="toself", fillcolor="rgba(209,72,63,0.10)", line={"width": 0},
                             hoverinfo="skip", showlegend=False), row=row, col=col)
    fig.add_trace(go.Scatter(x=INFORMATION, y=boundary.z, mode="lines", name="harm boundary",
                             line={"color": "#d1483f", "width": 2.0, "dash": "dash"},
                             showlegend=row == 1 and col == 1), row=row, col=col)
    fig.add_trace(go.Scatter(x=INFORMATION[:taken], y=z_paths[key][:taken], mode="lines+markers",
                             name="observed Z", line={"color": colour, "width": 2.6},
                             marker={"size": 8}, showlegend=row == 1 and col == 1),
                  row=row, col=col)
    if paths[key].stopped_at is not None:
        stop = paths[key].stop
        fig.add_trace(go.Scatter(x=[stop.information], y=[stop.z], mode="markers",
                                 name="crossing", marker={"symbol": "x", "size": 16, "color": "#111"},
                                 showlegend=False), row=row, col=col)
    fig.add_hline(y=0, line={"color": "rgba(0,0,0,0.3)", "dash": "dot"}, row=row, col=col)
    fig.update_yaxes(range=[-5.0, 4.0], gridcolor=h.GRID, row=row, col=col)
    fig.update_xaxes(title_text="information fraction" if row == 2 else "", gridcolor=h.GRID,
                     row=row, col=col)
fig.update_layout(height=640, template="plotly_white",
                  title="Safety monitoring of the 40 mg arm: pooled, and inside each age band",
                  legend={"orientation": "h", "y": 1.07, "x": 0.0},
                  margin={"l": 60, "r": 30, "t": 110, "b": 50})
fig

## 3. The stop

`MonitoringPath` carries the look that crossed, the threshold it crossed, and a
`LedgerLine` that names what is now true of the estimate — including the thing that is
easiest to forget.

In [ ]:
stopped = paths[("age_51_plus", "dose_40")]
print("decision      :", stopped.decision)
print("look          :", stopped.stopped_at + 1, "of", schedule.n_looks,
      f"(calendar week {LOOK_WEEKS[stopped.stop.look]})")
print("information   :", f"{stopped.information_used:.3f}")
print(f"effect        : {stopped.stop.effect:+.2f} {h.OUTCOME_UNIT} "
      f"(se {stopped.stop.se:.2f}, n {series[LOOK_WEEKS[stopped.stop.look]][('age_51_plus', 'dose_40')][2]})")
print(f"Z             : {stopped.stop.z:.3f} against a threshold of {stopped.threshold():.3f}")
line = stopped.ledger_line()
assert isinstance(line, LedgerLine)
print("\nledger:", line.statement)
print("assumption:", line.assumption.name)
print("  ", line.assumption.statement)
print("  challenged by:", line.assumption.challenged_by)

posterior = float(sps.norm.sf((HARM_MARGIN - stopped.stop.effect) / stopped.stop.se))
print(f"\nthe crossing restated as the protocol sentence: "
      f"P(40 mg is worse than control by more than {HARM_MARGIN:g} {h.OUTCOME_UNIT} | data) = "
      f"{posterior:.3f}  (the rule fires at {HARM_PROBABILITY:.2f})")

In [ ]:
week = LOOK_WEEKS[stopped.stop.look]
print(f"what the other monitors saw at the same review (calendar week {week}):")
for stratum, arm in CONTRASTS:
    if (stratum, arm) not in series[week] or arm != "dose_40":
        continue
    effect, se, n = series[week][(stratum, arm)]
    threshold = rules[(stratum, arm)].boundaries[0].z[stopped.stop.look]
    label = "all units" if stratum == "all" else h.STRATUM_LABEL[stratum]
    print(f"  {label:>10s}  effect {effect:+6.2f}  se {se:5.2f}  n {n:3d}  "
          f"Z {-effect / se:+6.2f}  threshold {threshold:+6.2f}  "
          f"{'CROSSED' if -effect / se <= threshold else 'continue'}")

The pooled monitor saw a Z of about +0.5 — an arm doing marginally *better* than the
control — at the same moment the oldest band crossed. Pre-specifying the stratum-level
contrasts is what made the difference; nothing about the analysis method did.

## 4. How many units were spared

In [ ]:
cell = trial.units[(trial.units["stratum"] == "age_51_plus") & (trial.units["arm"] == "dose_40")]
entry = cell["enrolled_week"].to_numpy()
stop_week = LOOK_WEEKS[stopped.stop.look]
randomized_by_stop = int((entry < stop_week).sum())
unit_weeks_taken = float(np.clip(np.minimum(stop_week, entry + h.FOLLOW_UP_WEEKS) - entry, 0, None).sum())
unit_weeks_planned = float(len(cell) * h.FOLLOW_UP_WEEKS)
print(f"planned for the 40 mg / 51+ cell : {len(cell)} units, {unit_weeks_planned:,.0f} unit-weeks")
print(f"actually randomized to it        : {randomized_by_stop} units")
print(f"actually dosed                   : {unit_weeks_taken:,.0f} unit-weeks "
      f"({unit_weeks_taken / unit_weeks_planned:.0%} of plan)")
print(f"never randomized to the arm      : {len(cell) - randomized_by_stop} units")
expected = operating_characteristics(
    rules[("age_51_plus", "dose_40")],
    h.intent_to_treat_contrast(40.0, "age_51_plus") / -float(
        rules[("age_51_plus", "dose_40")].boundaries[0].detail["se_at_full_information"]))
print(f"\nthe design expected to stop at {expected.expected_information:.2f} of the information "
      f"against this harm; this trial stopped at {stopped.information_used:.3f}.")

### Two weaker worlds

This arm crossed at the first review because the signal is large. Replay the same rule
against two worlds where the pressor effect is weaker — the only thing changed is the
`harm` parameter of the generating process — and the design behaves as the operating
characteristics in notebook 5 said it would: it stops one of them a review later, and
never stops the other.

In [ ]:
from dataclasses import replace

boundary_z = rules[("age_51_plus", "dose_40")].boundaries[0].z
print("boundary:", [round(v, 2) for v in boundary_z], "\n")
for harm_scale, label in ((26.0, "weaker"), (24.0, "weaker still")):
    world = replace(h.TRUTH, harm=harm_scale)
    other = h.trial(seed=20260821, truth=world)
    z_series, effect_series, se_series = [], [], []
    for week in LOOK_WEEKS:
        rows = h.contrast_frame(h.look_frame(other, week, endpoint="safety"), "dose_40",
                                stratum="age_51_plus")
        estimate = ols(rows, "change", "treated", h.ancova_covariates("age_51_plus"))
        z_series.append(-estimate.estimate / estimate.se)
        effect_series.append(estimate.estimate)
        se_series.append(estimate.se)
    other_path = monitor(rules[("age_51_plus", "dose_40")], z_series, effects=effect_series,
                         ses=se_series)
    true_harm = h.intent_to_treat_contrast(40.0, "age_51_plus", h.PRIMARY_WEEK, world)
    where = ("" if other_path.stop is None
             else f" at calendar week {LOOK_WEEKS[other_path.stop.look]}")
    print(f"{label:14s} true harm {true_harm:+5.2f} {h.OUTCOME_UNIT}  "
          f"Z {[round(v, 2) for v in z_series]}  -> {other_path.decision}{where}")

print(f"\n{'the trial above':14s} true harm "
      f"{h.intent_to_treat_contrast(40.0, 'age_51_plus'):+5.2f} {h.OUTCOME_UNIT}  "
      f"Z {[round(v, 2) for v in z_paths[('age_51_plus', 'dose_40')]]}  -> stop_harm at week 12")
print("\nA rule with a stated margin has a threshold, and worlds either side of it are")
print("treated differently. That is the design working, not a coincidence of this seed.")

## 5. The estimate at the stop is biased, and by how much

The ledger line says an estimate reported at the look that stopped the study is biased
away from the null. That is not a caveat, it is a computable number: simulate the
canonical joint distribution at the drift the trial actually saw, apply the same rule,
and average the estimate *conditional on stopping*.

In [ ]:
se_full = rules[("age_51_plus", "dose_40")].boundaries[0].detail["se_at_full_information"]
se_full = float(se_full)
final_frame = h.contrast_frame(h.look_frame(trial, h.TRIAL_WEEKS, endpoint="primary"),
                               "dose_40", stratum="age_51_plus")
final_estimate = ols(final_frame, "change", "treated", h.ancova_covariates("age_51_plus"))
truth = h.intent_to_treat_contrast(40.0, "age_51_plus")
print(f"truth (intent to treat, week 9–12)     : {truth:+.2f} {h.OUTCOME_UNIT}")
print(f"estimate at the stopping look          : {stopped.stop.effect:+.2f} "
      f"(n = {series[stop_week][('age_51_plus', 'dose_40')][2]})")
print(f"estimate on every unit that reached the primary window: "
      f"{final_estimate.estimate:+.2f} (n = {final_estimate.n})")

rng = np.random.default_rng(5)
N_SIMS = 20_000
increments = np.asarray(schedule.increments)
drift = -truth / se_full
steps = rng.normal(loc=drift * increments, scale=np.sqrt(increments), size=(N_SIMS, schedule.n_looks))
b = np.cumsum(steps, axis=1)
z = b / np.sqrt(np.asarray(schedule.information))
boundary_z = np.asarray(rules[("age_51_plus", "dose_40")].boundaries[0].z)
crossed = z <= boundary_z
first = np.where(crossed.any(axis=1), crossed.argmax(axis=1), -1)
stopping = first >= 0
look_se = se_full / np.sqrt(np.asarray(schedule.information))
reported = -z[np.arange(N_SIMS)[stopping], first[stopping]] * look_se[first[stopping]]
print(f"\n{stopping.mean():.1%} of simulated trials stop; conditional on stopping the reported")
print(f"harm averages {reported.mean():+.2f} {h.OUTCOME_UNIT} against a truth of {truth:+.2f} — "
      f"an exaggeration of {reported.mean() - truth:+.2f}.")

fig = h.figure("The estimate a stopped trial reports, against the truth",
               f"reported harm at the stopping look ({h.OUTCOME_UNIT})", "simulated trials",
               height=380)
fig.add_trace(go.Histogram(x=reported, nbinsx=60, marker_color="rgba(181,69,59,0.65)",
                           name="conditional on stopping"))
fig.add_vline(x=truth, line={"color": "#111", "dash": "dash"},
              annotation_text=f"truth {truth:.2f}")
fig.add_vline(x=float(reported.mean()), line={"color": "#d1483f"},
              annotation_text=f"mean {reported.mean():.2f}")
fig.add_vline(x=stopped.stop.effect, line={"color": "#2f7fd1", "dash": "dot"},
              annotation_text="this trial")
fig

The rule is right to have stopped, and the number it stopped on overstates the harm by
about 0.8 mmHg on average — a seventh of the true effect, and enough to matter if the
stopping estimate were the one that got quoted. Both statements are true, the ledger
line carries the second with the first, and the estimate that goes in the report is the
one from every unit that reached the primary window rather than the one from the look
that crossed.

## 6. The efficacy readout, on the arms that kept going

Enrollment did not run exactly to plan, so the realized information fractions are not
the planned ones. A boundary built from a spending function absorbs that:
`alpha_spending` re-solves the remaining thresholds against what accrued.

In [ ]:
EFFICACY_WEEKS = [18, 22, 26]
primary_accrual = h.accrual(trial, EFFICACY_WEEKS, endpoint="primary")
realized_information = tuple(float(x) for x in primary_accrual["share"])
print("planned  t =", (0.5, 0.75, 1.0))
print("realized t =", tuple(round(t, 3) for t in realized_information))
per_dose_alpha = 0.025 / 3
efficacy_schedule = LookSchedule(labels=tuple(f"week_{w}" for w in EFFICACY_WEEKS),
                                 information=realized_information)
efficacy = alpha_spending(per_dose_alpha, realized_information, family="obrien_fleming", side="upper")
futility = Boundary(kind="futility", side="lower", z=(-0.5, 0.3, 0.9), binding=False)
print("thresholds :", [round(z, 3) for z in efficacy.z])
print("alpha spent:", [round(s, 5) for s in efficacy.spent])

rows = []
for arm in ("dose_10", "dose_20"):
    rule = StoppingRule(name=f"efficacy|{arm}", looks=efficacy_schedule,
                        boundaries=(efficacy, futility))
    z_series, effect_series, se_series = [], [], []
    for week in EFFICACY_WEEKS:
        frame = h.contrast_frame(h.look_frame(trial, week, endpoint="primary"), arm)
        estimate = ols(frame, "change", "treated", h.ancova_covariates(None))
        z_series.append(-estimate.estimate / estimate.se)
        effect_series.append(estimate.estimate)
        se_series.append(estimate.se)
    path = monitor(rule, z_series, effects=effect_series, ses=se_series)
    rows.append({"arm": h.ARM_LABEL[arm], "decision": path.decision,
                 "stopped_week": "" if path.stop is None else EFFICACY_WEEKS[path.stop.look],
                 "z": [round(v, 2) for v in z_series],
                 "effect_at_stop": np.nan if path.stop is None else path.stop.effect})
print()
print(pd.DataFrame(rows).to_string(index=False))

Both surviving doses cross the efficacy boundary at the first look they are eligible
for, which is what a boundary built to be hard to cross early means when the effect is
genuinely large.

## 7. The readout

Estimates from every randomized unit that reached the primary window, with intervals
that carry their definition and their mass.

In [ ]:
graph = CausalGraph.from_edges(
    "age -> baseline; age -> sbp; baseline -> sbp; dose -> adherence; dose -> sbp; "
    "adherence -> sbp; sbp -> dropout", name="hyper3")
verdict = identify(graph, "dose", "sbp")
final = h.look_frame(trial, h.TRIAL_WEEKS, endpoint="primary")
rows = []
for arm in h.ARMS[1:]:
    for stratum in [None, *h.STRATA]:
        frame = h.contrast_frame(final, arm, stratum=stratum)
        estimate = ols(frame, "change", "treated", h.ancova_covariates(stratum))
        band: Interval = wald(-estimate.estimate, estimate.se, MASS)
        rows.append({"arm": h.ARM_LABEL[arm],
                     "stratum": "all units" if stratum is None else h.STRATUM_LABEL[stratum],
                     "reduction": -estimate.estimate, "se": estimate.se, "n": estimate.n,
                     "lower": band.lower, "upper": band.upper,
                     "truth": -h.intent_to_treat_contrast(h.DOSE[arm], stratum)})
readout = pd.DataFrame(rows)
print(f"reduction in SBP against the standard of care, {MASS:.0%} {band.definition} intervals")
print(f"identification: {verdict.verdict.status} by {verdict.route}, "
      f"adjustment set {verdict.adjustment_set or '{}'}")
print(readout.round(2).to_string(index=False))

In [ ]:
fig = h.figure("HYPER-3 readout: reduction in systolic pressure against the standard of care",
               f"reduction ({h.OUTCOME_UNIT}) — positive is better", "", height=520)
labels, order = [], []
for arm in reversed(h.ARMS[1:]):
    for stratum in reversed(["all units", *[h.STRATUM_LABEL[s] for s in h.STRATA]]):
        row = readout[(readout["arm"] == h.ARM_LABEL[arm]) & (readout["stratum"] == stratum)].iloc[0]
        labels.append(f"{h.ARM_LABEL[arm]} · {stratum}")
        order.append(row)
frame = pd.DataFrame(order)
frame["label"] = labels
colours = ["#5b6472" if s == "all units" else
           h.STRATUM_COLOR[{v: k for k, v in h.STRATUM_LABEL.items()}[s]] for s in frame["stratum"]]
fig.add_trace(go.Scatter(x=frame["reduction"], y=frame["label"], mode="markers",
                         marker={"size": 11, "color": colours},
                         error_x={"type": "data", "symmetric": False,
                                  "array": frame["upper"] - frame["reduction"],
                                  "arrayminus": frame["reduction"] - frame["lower"]},
                         name="estimate"))
fig.add_trace(go.Scatter(x=frame["truth"], y=frame["label"], mode="markers", name="truth",
                         marker={"symbol": "line-ns-open", "size": 14,
                                 "line": {"width": 2, "color": "#111"}}))
fig.add_vline(x=0, line={"color": "rgba(0,0,0,0.5)"})
fig.update_layout(margin={"l": 200, "r": 30, "t": 60, "b": 50})
fig

## 8. The heterogeneity is the finding

Three stratum-level estimates of the same nominal quantity. `meta.random_effects` pools
them and `meta.heterogeneity` says how much of the spread is real rather than sampling
noise. For 10 and 20 mg the answer is "none worth reporting". For 40 mg it is almost all
of it.

In [ ]:
for arm in h.ARMS[1:]:
    rows = readout[(readout["arm"] == h.ARM_LABEL[arm]) & (readout["stratum"] != "all units")]
    pooled = random_effects(rows["reduction"].to_numpy(), rows["se"].to_numpy(), mass=MASS)
    stats_ = heterogeneity(rows["reduction"].to_numpy(), rows["se"].to_numpy())
    assert isinstance(stats_, Heterogeneity)
    print(f"{h.ARM_LABEL[arm]:>8s}  pooled {pooled.estimate:+6.2f} "
          f"[{pooled.interval.lower:+.2f}, {pooled.interval.upper:+.2f}]  "
          f"tau {np.sqrt(pooled.tau2):5.2f}  Q {stats_.q:6.2f}  I2 {stats_.i2:5.1%}  "
          f"p(Q) {stats_.p_value:.4f}")

In [ ]:
rows = readout[(readout["arm"] == "40 mg") & (readout["stratum"] != "all units")]
pooled = random_effects(rows["reduction"].to_numpy(), rows["se"].to_numpy(), mass=MASS)
data = forest_data(rows["reduction"].to_numpy(), rows["se"].to_numpy(), pooled,
                   labels=list(rows["stratum"]), mass=MASS)
assert isinstance(data, ForestData)
figure = forest(data)
figure.update_layout(title="40 mg against the standard of care, by age band (random effects)",
                     height=340, template="plotly_white")
figure

The pooled random-effects estimate for 40 mg is close to zero with an interval wide
enough to be useless, and `I²` says nearly all of that width is real heterogeneity, not
noise. Reporting the pooled number alone would be the single most misleading thing this
trial could do.

## 9. Could the harm have been chance?

A permutation test asks it directly: hold the outcomes fixed, reshuffle the arm labels
within the oldest stratum, and see how often a contrast this extreme appears.

In [ ]:
rng = np.random.default_rng(17)
observed = float(final_estimate.estimate)
frame = final_frame.copy()
null_draws = np.empty(2000)
for i in range(null_draws.size):
    frame["treated"] = rng.permutation(frame["treated"].to_numpy())
    null_draws[i] = ols(frame, "change", "treated", h.ancova_covariates("age_51_plus")).estimate
p_value = float((np.abs(null_draws) >= abs(observed)).mean())
band = eti(null_draws, 0.95)
print(f"observed contrast {observed:+.2f} {h.OUTCOME_UNIT}")
print(f"permutation null: mean {null_draws.mean():+.3f}, 95% {band}")
print(f"two-sided p = {p_value:.4f} on {null_draws.size} permutations"
      f"{' (no permutation reached the observed value)' if p_value == 0.0 else ''}")

fig = h.figure("Permutation null for the 40 mg contrast in the oldest band",
               f"contrast under permuted labels ({h.OUTCOME_UNIT})", "permutations", height=360)
fig.add_trace(go.Histogram(x=null_draws, nbinsx=60, marker_color="rgba(90,100,115,0.55)",
                           name="permuted"))
fig.add_vline(x=observed, line={"color": "#d1483f", "width": 3},
              annotation_text=f"observed {observed:+.2f}")
fig

## 10. The ledger

Rule 4 of this repository: every number carries its provenance. Here is everything the
trial's conclusions rest on, as the lines the code emitted rather than as prose someone
wrote afterwards.

In [ ]:
ledger: list[LedgerLine] = [stopped.ledger_line()]
ledger.append(LedgerLine(
    kind="identification",
    statement=(f"the dose→sbp contrast is {verdict.verdict.status} by the {verdict.route} route "
               f"with adjustment set {sorted(verdict.adjustment_set) or '{}'} — dose is randomized"),
    detail={"graph_hash": verdict.graph_hash[:16]}))
ledger.append(LedgerLine(
    kind="analysis_population",
    statement=("every randomized unit reaching the primary window is analysed; units are not "
               "restricted to completers, because dropout is a descendant of the outcome"),
    detail={"n_randomized": str(trial.n_units), "n_analysed": str(len(final))}))
ledger.append(LedgerLine(
    kind="multiplicity",
    statement=(f"{len(rules)} harm contrasts were monitored without a multiplicity correction; "
               "the family-wise false-stop rate under the null is about 0.085"),
    detail={"contrasts": str(len(rules)), "per_contrast": "0.001 to 0.015"}))
ledger.append(LedgerLine(
    kind="transport",
    statement=("the trial-population average does not transport to an older prescribing "
               "population; the effect is s-admissible given age and must be re-standardized"),
    detail={"s_admissible_set": "age"}))
for entry in ledger:
    print(f"[{entry.kind}]")
    print(f"   {entry.statement}")
    if entry.assumption is not None:
        print(f"   assumption: {entry.assumption.name} ({entry.assumption.state})")
    if entry.detail:
        print(f"   {entry.detail}")

## 11. The readout, as a report anyone can regenerate

The last step of a trial is the document, and a document assembled by hand is a
document that drifts from the analysis. `axiom.report` takes a template that holds
**no data** — every block names a context key — so the same template can be re-run when
the database locks again, and its content hash says the layout did not move.

Two things it cannot do: show a surface without its band (a `ResponseBand` source is
drawn through `viz`, which has no such mode), and show an estimate without its interval
(a `Metric` whose source carries one prints it with its definition and mass).

In [ ]:
from axiom.report import ReportBuilder, Theme, missing, render, write

trial_theme = Theme(
    name="hyper3", accent_color="#b5453b", muted_color="#5b6472",
    palette=("#b5453b", "#2f7fd1", "#3aa17e", "#c9a227"), figure_height=320.0,
)
template = (
    ReportBuilder("hyper3_readout", "HYPER-3 — dose-finding readout")
    .theme(trial_theme)
    .subtitle("Database lock {as_of}. Intervals are {mass:.0%} unless stated.")
    .footer("Template {template_hash} · regenerate with axiom.report.write")
    .section("Decision", summary="What the trial concluded, and what it rests on.")
    .paragraph("**20 mg** goes forward. **40 mg is stopped in adults over 50**, on a harm "
               "boundary crossed at the first scheduled review.")
    .metric("dose_20_all", "20 mg, all units", unit="mmHg")
    .metric("dose_40_oldest", "40 mg, 51+", unit="mmHg")
    .metric("exposure_saved", "Unit-weeks of 40 mg avoided")
    .section("Safety monitoring", summary="The stop, and what the pooled monitor saw instead.")
    .table("monitoring", caption="Every monitored contrast at close", max_rows=12)
    .section("Dose response", summary="Fitted per age band; every curve carries its band.")
    .figure("oldest_curve", caption="Oldest band, {family} fit — the reversal above 20 mg")
    .section("Assumptions")
    .ledger("ledger", caption="Everything above rests on these", show_detail=True)
    .build()
)
print("template hash:", template.content_hash()[:16])
print("sources      :", template.sources())

The context is assembled from objects the earlier sections already produced, plus one
`ResponseBand` refitted here so the figure carries uncertainty rather than a mean line.

In [ ]:
from axiom.core import BASES, D, Outcome, Treatment, wald
from axiom.data import Panel, RoleMap
from axiom.surface import SplineKernel, SurfaceSpec, fit as surface_fit, response_band

BASES.declare("mass", symbol="M")
reduction = Outcome(name="reduction", dimension=D.outcome, unit=h.OUTCOME_UNIT, aggregation="mean")
drug = Treatment(name="dose", dimension=D.mass, unit=h.DOSE_UNIT)
oldest = final[final["stratum"] == "age_51_plus"].copy()
oldest["reduction"] = -oldest["change"]
oldest["t"] = 0
oldest_panel = Panel(
    oldest[["unit", "t", "reduction", "dose"]],
    RoleMap(unit="unit", time="t", outcome=("reduction", reduction), treatments={"dose": drug}),
)
flexible = SurfaceSpec(
    name="oldest", treatments=(drug,), outcome=reduction,
    kernels={"dose": SplineKernel(reference_dose=40.0, amplitude_scale=10.0,
                                  knots=(10.0, 20.0, 30.0))},
    intercept="shared", noise_scale=6.0,
)
oldest_fit = surface_fit(flexible, oldest_panel, backend="laplace", draws=600, chains=1,
                         seed=20260821)
oldest_band = response_band(oldest_fit, "dose", n_grid=25, mass=0.95)
print("band:", oldest_band.label(), "over", oldest_band.n_draws, "draws;",
      "widest interval", f"{max(oldest_band.width):.2f}", h.OUTCOME_UNIT)

cell_units = trial.units[(trial.units["stratum"] == "age_51_plus")
                         & (trial.units["arm"] == "dose_40")]
planned_weeks = len(cell_units) * h.FOLLOW_UP_WEEKS
context = {
    "as_of": "2026-08-21",
    "mass": 0.95,
    "template_hash": template.content_hash()[:12],
    "family": "natural cubic spline",
    "dose_20_all": wald(
        -ols(h.contrast_frame(final, "dose_20"), "change", "treated",
             h.ancova_covariates(None)).estimate,
        ols(h.contrast_frame(final, "dose_20"), "change", "treated",
            h.ancova_covariates(None)).se, 0.95),
    "dose_40_oldest": wald(-final_estimate.estimate, final_estimate.se, 0.95),
    "exposure_saved": planned_weeks - unit_weeks_taken,
    "monitoring": monitoring[["stratum", "arm", "decision", "stopped_week", "lowest_z_seen"]],
    "oldest_curve": oldest_band,
    "ledger": ledger,
}
print("missing:", missing(template, context))

In [ ]:
import pathlib
import tempfile

from axiom.core import is_failure

out = pathlib.Path(tempfile.mkdtemp())
for fmt in ("html", "pptx", "pdf"):
    written = write(template, context, str(out / f"hyper3_readout.{fmt}"), fmt)
    if is_failure(written):
        print(f"{fmt:5s} refused: {written.reason}")
    else:
        print(f"{fmt:5s} -> {pathlib.Path(written).stat().st_size:>9,} bytes")

document = render(template, context, "html")
assert isinstance(document, str)
print("\nthe band survived into the document:", document.count('"fill":"toself"'), "filled trace(s)")
print("the intervals survived            :",
      document.count("(95% WALD)") + document.count("(95% ETI)"), "shown")
print("the template hash is in the file  :", template.content_hash() in document)

Re-running this after a second database lock is the same call with a different context.
The template's content hash is unchanged, which is the auditable claim that the layout —
what is shown, and next to what — did not move between the two readouts.

## What HYPER-3 concluded

- **20 mg is the dose to take forward**, at a reduction of about 4–5 mmHg against the
  standard of care with no age band showing harm. 10 mg is similar and slightly weaker.
- **40 mg is stopped in adults over 50.** It crossed at the *first* scheduled safety
  review, on three eighths of the information and 33 of the 75 units that contrast would
  eventually have had — 9 of the 28 units planned for the arm were never randomized to
  it, and the arm delivered 119 of a planned 672 unit-weeks of drug. Pooled over the
  trial population the same arm never looked like anything worth stopping, at any review;
  at the review that stopped it, the pooled monitor was reading `Z = +0.9`, an arm doing
  slightly *better* than control. The design saw the harm only because the age strata
  were pre-specified as monitored contrasts.
- **The number it stopped on overstates the harm.** Simulation at the realized drift
  puts the conditional exaggeration at about 0.8 mmHg — a seventh of the effect. The
  ledger line says so, and the reported estimate is the one from every unit that reached
  the primary window.
- **The pooled 40 mg estimate should not be reported alone.** `I²` puts almost all the
  spread across age bands down to real heterogeneity rather than noise.
- **Twelve contrasts were monitored and no multiplicity correction was applied.** The
  family-wise false-stop rate is about one trial in twelve. That is in the ledger, it was
  chosen deliberately, and a stopped arm is a signal to investigate rather than a verdict.
- The next trial should not repeat 0/10/20/40: notebook 4's `design_to_identify` puts the
  doses elsewhere, and notebook 2's transport verdict says it should be run — or at least
  re-standardized — for the older population that will actually take the drug.